# Wykrywanie oszustw kartowych — eksploracyjna analiza danych (EDA)

**Projekt zaliczeniowy — kurs Data Science, grupa jdszr24-2**

Zbiór: `credit_card_fraud_10k.csv` — 10 000 transakcji kartami płatniczymi.
Cel projektu: zbudować model klasyfikacyjny wykrywający transakcje oszukańcze (`is_fraud = 1`).

| Kolumna | Opis |
|---|---|
| `transaction_id` | identyfikator transakcji |
| `amount` | kwota transakcji |
| `transaction_hour` | godzina transakcji (0–23) |
| `merchant_category` | kategoria sprzedawcy |
| `foreign_transaction` | czy transakcja zagraniczna (0/1) |
| `location_mismatch` | czy lokalizacja niezgodna z profilem klienta (0/1) |
| `device_trust_score` | ocena zaufania urządzenia (0–100) |
| `velocity_last_24h` | liczba transakcji w ostatnich 24 h |
| `cardholder_age` | wiek posiadacza karty |
| `is_fraud` | **zmienna celu**: 1 = oszustwo |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
FIG_DIR = "../reports/figures"

df = pd.read_csv("../data/raw/credit_card_fraud_10k.csv")
df.head()

## 1. Podstawowe informacje o zbiorze

In [ ]:
df.info()

In [ ]:
df.describe().T.round(2)

In [ ]:
print("Braki danych w kolumnach:")
print(df.isna().sum())
print(f"\nDuplikaty wierszy: {df.duplicated().sum()}")
print(f"Duplikaty transaction_id: {df['transaction_id'].duplicated().sum()}")

Zbiór jest kompletny — brak braków danych i duplikatów, typy kolumn poprawne.
Nie ma potrzeby imputacji ani czyszczenia.

## 2. Rozkład zmiennej celu (`is_fraud`)

In [ ]:
class_counts = df["is_fraud"].value_counts()
fraud_rate = df["is_fraud"].mean()
print(class_counts)
print(f"\nUdział fraudów: {fraud_rate:.2%}")

fig, ax = plt.subplots(figsize=(5, 4))
class_counts.plot(kind="bar", color=["#4878CF", "#D65F5F"], ax=ax)
ax.set_xticklabels(["prawidłowa (0)", "oszustwo (1)"], rotation=0)
ax.set_ylabel("liczba transakcji")
ax.set_title("Rozkład klas — silne niezbalansowanie (1,5% fraudów)")
for i, v in enumerate(class_counts):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/01_rozklad_klas.png", dpi=120)
plt.show()

Tylko **1,5%** transakcji to oszustwa — to kluczowa cecha problemu.
Wniosek dla modelowania: accuracy jest bezużyteczne (model zawsze przewidujący
„brak fraudu" miałby 98,5% accuracy), trzeba używać metryk precision/recall/PR-AUC
oraz ważenia klas.

## 3. Rozkłady zmiennych liczbowych

In [ ]:
num_cols = ["amount", "transaction_hour", "device_trust_score",
            "velocity_last_24h", "cardholder_age"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, num_cols):
    sns.histplot(df[col], bins=40, ax=ax, color="#4878CF")
    ax.set_title(col)
axes.flat[-1].axis("off")
fig.suptitle("Rozkłady zmiennych liczbowych", y=1.02)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/02_rozklady_zmiennych.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Częstość oszustw w podziale na cechy

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

df.groupby("merchant_category")["is_fraud"].mean().sort_values().plot(
    kind="barh", ax=axes[0, 0], color="#D65F5F")
axes[0, 0].set_title("Udział fraudów wg kategorii sprzedawcy")
axes[0, 0].set_xlabel("fraud rate")

df.groupby("transaction_hour")["is_fraud"].mean().plot(
    kind="bar", ax=axes[0, 1], color="#D65F5F")
axes[0, 1].set_title("Udział fraudów wg godziny transakcji")
axes[0, 1].set_ylabel("fraud rate")

df.groupby("foreign_transaction")["is_fraud"].mean().plot(
    kind="bar", ax=axes[1, 0], color="#D65F5F")
axes[1, 0].set_title("Udział fraudów: transakcja zagraniczna")
axes[1, 0].set_xticklabels(["krajowa", "zagraniczna"], rotation=0)

df.groupby("location_mismatch")["is_fraud"].mean().plot(
    kind="bar", ax=axes[1, 1], color="#D65F5F")
axes[1, 1].set_title("Udział fraudów: niezgodność lokalizacji")
axes[1, 1].set_xticklabels(["zgodna", "niezgodna"], rotation=0)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/03_fraud_rate_wg_cech.png", dpi=120)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col in zip(axes, ["amount", "device_trust_score", "velocity_last_24h"]):
    sns.boxplot(data=df, x="is_fraud", y=col, hue="is_fraud",
                palette=["#4878CF", "#D65F5F"], legend=False, ax=ax)
    ax.set_title(f"{col} vs is_fraud")
    ax.set_xticklabels(["prawidłowa", "oszustwo"])
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/04_boxploty.png", dpi=120)
plt.show()

## 5. Korelacje

In [ ]:
corr = df.drop(columns=["transaction_id"]).select_dtypes("number").corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Macierz korelacji (Pearson)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/05_korelacje.png", dpi=120)
plt.show()

print("Korelacja cech z is_fraud (malejąco wg wartości bezwzględnej):")
print(corr["is_fraud"].drop("is_fraud").reindex(
    corr["is_fraud"].drop("is_fraud").abs().sort_values(ascending=False).index).round(3))

## 6. Wnioski z EDA

1. **Dane są czyste** — bez braków i duplikatów, nie wymagają imputacji.
2. **Klasy silnie niezbalansowane (1,5% fraudów)** — w modelowaniu użyjemy
   `class_weight='balanced'` i metryk odpornych na niezbalansowanie (PR-AUC, recall).
3. Największy związek z fraudem wykazują (kierunki zgodne z intuicją):
   `location_mismatch`, `foreign_transaction`, `device_trust_score` (ujemnie),
   `velocity_last_24h` oraz `amount`.
4. `merchant_category` różnicuje fraud rate umiarkowanie — zakodujemy ją one-hot
   i pozwolimy modelowi zdecydować o jej przydatności.
5. `transaction_id` to czysty identyfikator — do usunięcia przed modelowaniem.

Modelowanie: patrz `02_model.ipynb`.